# 🌉 Module 11.5: Sim-to-Real Transfer for Vision RL

## Bridging the Reality Gap — From Synthetic Pixels to Real-World Deployment

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FlashVision/VisionRL/blob/main/Module_11_Advanced_Topics/05_Sim_To_Real_Transfer/sim_to_real_transfer.ipynb)

---

## 🎯 Learning Objectives

1. **Understand the reality gap** — why policies trained on synthetic images fail on real-world data
2. **Formalize domain shift** — mathematical framework for distribution mismatch between sim and real
3. **Implement domain randomization** — create diverse synthetic training data that covers real-world variation
4. **Build domain adaptation** — adversarial feature alignment to learn domain-invariant representations
5. **Measure transfer quality** — MMD, feature-space t-SNE, transfer gap analysis
6. **Compare approaches** — no transfer vs. domain randomization vs. domain adaptation

---

In [ ]:
# Install dependencies (Google Colab compatible)
!pip install numpy matplotlib torch torchvision scikit-learn -q

In [ ]:
# Download REAL open-source datasets for advanced RL (TINY - under 100MB)
import torchvision

# Omniglot: 1623 characters from 50 alphabets (perfect for meta-learning, only 13MB!)
try:
    omniglot = torchvision.datasets.Omniglot(root='./data', download=True)
    print(f"✅ Omniglot: {len(omniglot)} real handwritten characters (13MB)")
    print("   50 different alphabets - perfect for few-shot/meta-learning!")
except Exception as e:
    print(f"⚠️ Omniglot: {e}, using MNIST split into tasks")

# CIFAR-10 for curriculum learning and multi-agent tasks
cifar10 = torchvision.datasets.CIFAR10(root='./data', train=True, download=True)
print(f"✅ CIFAR-10: {len(cifar10)} real photos (170MB, likely already cached)")

# FashionMNIST as second domain for transfer learning (instead of STL-10!)
fashion = torchvision.datasets.FashionMNIST(root='./data', train=True, download=True)
print(f"✅ FashionMNIST: {len(fashion)} real clothing images (30MB)")
print("   Use MNIST→FashionMNIST as sim-to-real domain shift!")

print("\n📦 Total new download: ~43MB (Omniglot + FashionMNIST)")
print("   NO STL-10 (2.6GB) needed! FashionMNIST works great for domain transfer.")

## Deep Derivation: Sim-to-Real Transfer and Domain Adaptation Theory

### Step 1: Domain Shift Formalization
Let $\mathcal{D}_S = (X_S, p_S)$ be the source (simulation) domain and $\mathcal{D}_T = (X_T, p_T)$ the target (real) domain. The domain shift is quantified by:

**Total variation distance:**
$$d_{TV}(p_S, p_T) = \sup_{A \in \mathcal{F}} |p_S(A) - p_T(A)| = \frac{1}{2}\int |p_S(x) - p_T(x)| dx$$

**$\mathcal{H}$-divergence (proxy $\mathcal{A}$-distance):**
$$d_\mathcal{H}(p_S, p_T) = 2\sup_{h \in \mathcal{H}} \left|E_{x \sim p_S}[\mathbb{1}[h(x)=1]] - E_{x \sim p_T}[\mathbb{1}[h(x)=1]]\right|$$

**Empirical estimation:** Train a domain classifier $h$ to distinguish source from target. The proxy $\mathcal{A}$-distance:
$$\hat{d}_\mathcal{A} = 2(1 - 2\epsilon_h)$$

where $\epsilon_h$ is the classification error. If $\epsilon_h \approx 0.5$ (cannot distinguish domains), $\hat{d}_\mathcal{A} \approx 0$ (no domain shift). $\blacksquare$

### Step 2: Ben-David's Domain Adaptation Bound
**Theorem (Ben-David et al., 2010):** For any hypothesis $h \in \mathcal{H}$:
$$\epsilon_T(h) \leq \epsilon_S(h) + \frac{1}{2}d_{\mathcal{H}\Delta\mathcal{H}}(\mathcal{D}_S, \mathcal{D}_T) + \lambda^*$$

where $\epsilon_T(h)$ = target error, $\epsilon_S(h)$ = source error, and $\lambda^* = \min_{h^* \in \mathcal{H}} [\epsilon_S(h^*) + \epsilon_T(h^*)]$ is the optimal joint error.

**Derivation:**
$$\epsilon_T(h) = \epsilon_T(h) - \epsilon_S(h) + \epsilon_S(h) \leq |\epsilon_T(h) - \epsilon_S(h)| + \epsilon_S(h)$$

For the divergence term, consider $h, h' \in \mathcal{H}$:
$$|\epsilon_T(h) - \epsilon_S(h)| = |E_{p_T}[\ell(h)] - E_{p_S}[\ell(h)]| \leq \frac{1}{2}d_{\mathcal{H}\Delta\mathcal{H}}$$

where $\mathcal{H}\Delta\mathcal{H} = \{h \oplus h' : h, h' \in \mathcal{H}\}$ is the symmetric difference hypothesis class.

**Implication for sim-to-real:** Minimize source error AND domain divergence simultaneously! $\blacksquare$

### Step 3: Domain Randomization (Mathematical Framework)
Train on randomized source domains $\{\mathcal{D}_S^{(i)}\}_{i=1}^M$ with random parameters $\xi \sim p(\xi)$:
$$\min_\theta E_{\xi \sim p(\xi)}\left[\mathcal{L}(\theta; \mathcal{D}_S^{(\xi)})\right]$$

**Randomization parameters for vision:**
- Lighting: direction $\ell \sim \text{Uniform}(S^2)$, intensity $I \sim \text{Uniform}[0.5, 2.0]$
- Texture: color $(r,g,b) \sim \text{Uniform}([0,1]^3)$
- Camera: intrinsics $f \sim \mathcal{N}(\mu_f, \sigma_f^2)$, pose $T \sim SE(3)$

**Derivation of why randomization works:** If the real domain $p_T$ lies within the convex hull of randomized domains:
$$p_T \in \text{conv}\{p_S^{(\xi)} : \xi \in \Xi\}$$

then $\exists$ weights $w(\xi)$ such that $p_T = \int w(\xi) p_S^{(\xi)} d\xi$, and:
$$\epsilon_T(h) \leq \int w(\xi) \epsilon_S^{(\xi)}(h) d\xi \leq \max_\xi \epsilon_S^{(\xi)}(h)$$

Minimizing worst-case source error implicitly minimizes target error. $\blacksquare$

### Step 4: Adversarial Domain Adaptation (DANN)
Learn domain-invariant features via a gradient reversal layer:

**Feature extractor:** $G_f : X \to \mathbb{R}^d$ (shared)
**Task classifier:** $G_y : \mathbb{R}^d \to \mathcal{Y}$ (task-specific)
**Domain discriminator:** $G_d : \mathbb{R}^d \to \{0,1\}$ (source vs. target)

**Minimax objective:**
$$\min_{G_f, G_y} \max_{G_d} \underbrace{E_{(x,y)\sim\mathcal{D}_S}[\log G_y(G_f(x), y)]}_{\text{task loss on source}} - \underbrace{\lambda\left(E_{x\sim\mathcal{D}_S}[\log G_d(G_f(x))] + E_{x\sim\mathcal{D}_T}[\log(1-G_d(G_f(x)))]\right)}_{\text{domain confusion}}$$

**Gradient reversal layer (GRL):** During forward pass: identity. During backward pass: negate gradient.
$$\text{GRL}(x) = x, \quad \frac{\partial \text{GRL}}{\partial x} = -I$$

**Derivation of equivalence to minimax:** With GRL, we can use standard SGD. The feature extractor receives:
$$\nabla_{G_f} = \nabla_{G_f}\mathcal{L}_{\text{task}} - \lambda\nabla_{G_f}\mathcal{L}_{\text{domain}}$$

This simultaneously minimizes task loss and maximizes domain confusion — exactly the minimax objective above. $\blacksquare$

### Step 5: Visual Domain Randomization for RL Policies
For RL policies trained in simulation:
$$\pi^*(a|o) = \arg\max_\pi E_{\xi}\left[V^\pi_{\xi}(s_0)\right]$$

**Robust MDP formulation:** The domain-randomized policy is the solution to:
$$\max_\pi \min_{P \in \mathcal{P}} E_P\left[\sum_t \gamma^t r_t\right]$$

where $\mathcal{P}$ is the uncertainty set of transition models.

**Derivation of the uncertainty set:** If simulation parameters $\xi$ induce transitions $T_\xi(s'|s,a)$, then:
$$\mathcal{P} = \{T_\xi : \xi \in \Xi\} \cup \{T_{\text{real}}\}$$

The robust policy maximizes worst-case performance:
$$V^{\text{robust}}(s) = \max_\pi \min_{T \in \mathcal{P}} E_T\left[\sum_{t=0}^\infty \gamma^t r_t \middle| s_0 = s, \pi\right]$$

**Iyengar's result (2005):** The robust Bellman equation:
$$V(s) = \max_a \min_{T \in \mathcal{P}} \left[R(s,a) + \gamma \sum_{s'} T(s'|s,a) V(s')\right]$$

converges under value iteration if $\mathcal{P}$ is an $(s,a)$-rectangular uncertainty set. $\blacksquare$

### Step 6: CycleGAN for Visual Domain Translation
Translate simulation images to look realistic without paired examples.

**Cycle-consistency loss:**
$$\mathcal{L}_{\text{cyc}}(G, F) = E_{x\sim p_S}[\|F(G(x)) - x\|_1] + E_{y\sim p_T}[\|G(F(y)) - y\|_1]$$

**Derivation of why cycle-consistency works:** Without it, $G$ could map all inputs to a single realistic image (mode collapse). Cycle-consistency enforces:
$$F \circ G \approx \text{Id}_S, \quad G \circ F \approx \text{Id}_T$$

This means $G$ is approximately invertible → $G$ must be approximately a bijection → each simulation image maps to a unique real image.

**Formal information-theoretic argument:** Cycle-consistency bounds the mutual information:
$$I(X; G(X)) \geq H(X) - E[\|F(G(X)) - X\|_1] / \sigma$$

High mutual information ensures $G$ preserves the structural content of simulation images while changing visual appearance. $\blacksquare$

### Step 7: Measuring Transfer Success — Reality Gap Metrics
Define the **reality gap** as the performance drop:
$$\Delta = \text{Performance}_{\text{real}} - \text{Performance}_{\text{sim}}$$

**Transfer ratio:**
$$\rho = \frac{\text{Performance}_{\text{sim} \to \text{real}}}{\text{Performance}_{\text{real} \to \text{real}}}$$

$\rho = 1$ means perfect transfer. $\rho = 0$ means complete failure.

**Derivation of expected transfer ratio under domain randomization:** If each randomization parameter independently contributes to the gap:
$$\rho \approx \prod_{i=1}^d (1 - \delta_i)$$

where $\delta_i$ is the marginal gap from parameter $i$. With $d$ parameters each contributing $\delta$ gap:
$$\rho \approx (1-\delta)^d \approx e^{-d\delta}$$

This exponential decay explains why high-dimensional domain randomization requires each parameter to be well-calibrated ($\delta \to 0$). Even small per-parameter gaps compound multiplicatively. $\blacksquare$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

plt.rcParams.update({
    'figure.dpi': 100,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})

---

## 1. Introduction to Sim-to-Real Transfer

### The Reality Gap

In vision-based RL, collecting training data in the real world is **expensive**, **slow**, and **dangerous**. Simulation offers a compelling alternative: unlimited, labelled, cheap data. But there is a catch — the **reality gap**.

A policy $\pi$ trained on simulated images $x_{\text{sim}} \sim P_{\text{sim}}(x)$ may fail catastrophically when deployed on real images $x_{\text{real}} \sim P_{\text{real}}(x)$ because:

- Simulated textures are too clean (no scratches, dirt, wear)
- Lighting is unrealistic (uniform vs. complex real illumination)
- Noise characteristics differ (Gaussian in sim vs. structured sensor noise)
- Color distributions shift (renderer gamut vs. camera gamut)

### Why Sim-to-Real?

| Aspect | Real Data | Simulated Data |
|--------|-----------|----------------|
| Cost per sample | \$\$\$ | ~Free |
| Labels | Manual | Automatic |
| Safety | Risk of damage | Zero risk |
| Volume | Limited | Unlimited |
| Diversity | Fixed setup | Infinite variation |
| Domain gap | None | **Must be bridged** |

The goal of sim-to-real transfer is to train in simulation and deploy in reality, closing the gap via **domain randomization** and **domain adaptation**.

In [ ]:
# --- Visualize the reality gap concept ---

def make_clean_shape_image(size=64, shape='circle', bg_val=0.1, fg_val=0.9):
    """Sim-domain: clean geometric shape on uniform background."""
    img = np.full((size, size), bg_val, dtype=np.float32)
    cx, cy = size // 2, size // 2
    if shape == 'circle':
        yy, xx = np.ogrid[:size, :size]
        mask = (xx - cx)**2 + (yy - cy)**2 <= (size // 4)**2
        img[mask] = fg_val
    elif shape == 'square':
        s = size // 4
        img[cy - s:cy + s, cx - s:cx + s] = fg_val
    elif shape == 'triangle':
        for row in range(size):
            half_w = max(0, int((size // 4) * (row - cy + size // 4) / (size // 4)))
            if cy - size // 4 <= row <= cy + size // 4 - 1:
                left = max(0, cx - half_w)
                right = min(size, cx + half_w)
                img[row, left:right] = fg_val
    return img


def add_real_effects(img, noise_std=0.12, blur_k=3, brightness_shift=0.0,
                     texture_freq=8, texture_amp=0.08):
    """Real-domain: add noise, blur, lighting gradient, texture."""
    h, w = img.shape
    out = img.copy()
    yy, xx = np.mgrid[:h, :w]
    gradient = (xx / w) * 0.15 + (yy / h) * 0.10
    out = out + gradient + brightness_shift
    texture = texture_amp * np.sin(2 * np.pi * texture_freq * xx / w) * \
              np.cos(2 * np.pi * texture_freq * yy / h)
    out = out + texture
    out = out + np.random.randn(h, w).astype(np.float32) * noise_std
    if blur_k > 1:
        from scipy.ndimage import uniform_filter
        out = uniform_filter(out, size=blur_k)
    return np.clip(out, 0, 1)


fig, axes = plt.subplots(2, 3, figsize=(12, 8))
shapes = ['circle', 'square', 'triangle']
for i, shape in enumerate(shapes):
    sim_img = make_clean_shape_image(64, shape)
    real_img = add_real_effects(sim_img)
    axes[0, i].imshow(sim_img, cmap='gray', vmin=0, vmax=1)
    axes[0, i].set_title(f'Sim: {shape.capitalize()}', fontweight='bold')
    axes[0, i].axis('off')
    axes[1, i].imshow(real_img, cmap='gray', vmin=0, vmax=1)
    axes[1, i].set_title(f'Real: {shape.capitalize()}', fontweight='bold')
    axes[1, i].axis('off')

fig.text(0.02, 0.72, 'SIM\nDomain', fontsize=14, fontweight='bold',
         color='#2196F3', va='center', ha='center')
fig.text(0.02, 0.28, 'REAL\nDomain', fontsize=14, fontweight='bold',
         color='#F44336', va='center', ha='center')
fig.suptitle('The Reality Gap: Same Shapes, Different Distributions',
             fontsize=15, fontweight='bold', y=0.98)
plt.tight_layout(rect=[0.05, 0, 1, 0.95])
plt.show()

---

## 2. Mathematical Framework

### 2.1 Domain Shift Formalization

Let $\mathcal{X}$ be the image space and $\mathcal{Y}$ the label/action space. We have two distributions:

$$P_{\text{sim}}(x) \neq P_{\text{real}}(x) \quad \text{(marginal distributions differ)}$$

but we desire:

$$P(y \mid x) \approx \text{const across domains (label semantics are shared)}$$

Under **covariate shift**, the conditional $P(y|x)$ remains the same, but the input distribution changes. The challenge is learning a representation $\phi(x)$ such that:

$$P_{\text{sim}}(\phi(x)) \approx P_{\text{real}}(\phi(x))$$

### 2.2 Domain Randomization

Instead of adapting to a specific real domain, we **randomize** simulation parameters $\xi$ (lighting, texture, noise, color, etc.) drawn from a distribution $\Xi$:

$$\pi^* = \arg\max_{\pi} \; \mathbb{E}_{\xi \sim \Xi}\left[\mathbb{E}_{\pi}\left[\sum_{t=0}^{T} \gamma^t r_t \;\middle|\; \xi\right]\right]$$

The key insight: if $\Xi$ is broad enough, the real world becomes just **another randomization** the policy has already handled.

### 2.3 Domain Adaptation Loss

In domain adaptation, we jointly optimize the task objective and a domain alignment term:

$$\mathcal{L}_{\text{adapt}} = \mathcal{L}_{\text{task}}(\theta_f, \theta_y) + \lambda \, \mathcal{L}_{\text{domain}}(\theta_f, \theta_d)$$

where:
- $\theta_f$: feature extractor parameters (shared)
- $\theta_y$: task head parameters (classifier / policy)
- $\theta_d$: domain discriminator parameters
- $\lambda$: trade-off hyperparameter

With **adversarial** adaptation (gradient reversal), the feature extractor learns to **fool** the domain discriminator:

$$\min_{\theta_f, \theta_y} \max_{\theta_d} \; \mathcal{L}_{\text{task}} - \lambda \, \mathcal{L}_{\text{domain}}$$

### 2.4 Maximum Mean Discrepancy (MMD)

MMD measures the distance between two distributions in a reproducing kernel Hilbert space $\mathcal{H}$:

$$\text{MMD}^2(P, Q) = \left\| \mu_P - \mu_Q \right\|_{\mathcal{H}}^2 = \mathbb{E}_{x,x'\sim P}[k(x,x')] - 2\mathbb{E}_{x\sim P, y\sim Q}[k(x,y)] + \mathbb{E}_{y,y'\sim Q}[k(y,y')]$$

Using a Gaussian RBF kernel $k(x,y) = \exp\left(-\frac{\|x - y\|^2}{2\sigma^2}\right)$, MMD = 0 iff $P = Q$.

### 2.5 Invariant Feature Learning

The goal is to learn $\phi: \mathcal{X} \to \mathcal{Z}$ such that the induced feature distributions align:

$$\min_{\phi} \; \text{MMD}^2\big(\phi(X_{\text{sim}}),\; \phi(X_{\text{real}})\big) + \mathcal{L}_{\text{task}}\big(\phi(X_{\text{sim}}),\; Y_{\text{sim}}\big)$$

This yields features that are **discriminative** for the task but **invariant** to the domain.

In [ ]:
# --- Visualize domain shift as distribution mismatch ---

np.random.seed(SEED)
n = 800

sim_features = np.random.randn(n, 2).astype(np.float32) * 0.8 + np.array([2.0, 2.0])
real_features = np.random.randn(n, 2).astype(np.float32) * 1.1 + np.array([4.5, 4.0])

rotation = np.array([[np.cos(0.25), -np.sin(0.25)],
                     [np.sin(0.25),  np.cos(0.25)]])
adapted_features = (sim_features - sim_features.mean(0)) @ rotation * 0.95 + real_features.mean(0)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, title, sim_f, real_f in [
    (axes[0], 'Before Transfer\n(Large Domain Gap)', sim_features, real_features),
    (axes[1], 'Domain Randomization\n(Broadened Sim Distribution)', 
     np.vstack([sim_features, sim_features + np.random.randn(n, 2) * 1.5]),
     real_features),
    (axes[2], 'Domain Adaptation\n(Aligned Features)', adapted_features, real_features),
]:
    ax.scatter(sim_f[:, 0], sim_f[:, 1], alpha=0.3, s=8, c='#2196F3', label='Sim')
    ax.scatter(real_f[:, 0], real_f[:, 1], alpha=0.3, s=8, c='#F44336', label='Real')
    ax.set_title(title, fontweight='bold')
    ax.legend(markerscale=3, fontsize=10)
    ax.set_xlabel('Feature dim 1')
    ax.set_ylabel('Feature dim 2')
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)

fig.suptitle('Domain Shift in Feature Space', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# --- Compute MMD between distributions ---

def gaussian_kernel(x, y, sigma=1.0):
    """Gaussian RBF kernel."""
    x.shape[0]
    y.shape[0]
    x.shape[1]
    x = x.unsqueeze(1)  # (n, 1, d)
    y = y.unsqueeze(0)  # (1, m, d)
    return torch.exp(-torch.sum((x - y) ** 2, dim=2) / (2 * sigma ** 2))


def compute_mmd(x, y, sigma=1.0):
    """Unbiased estimate of MMD^2."""
    kxx = gaussian_kernel(x, x, sigma)
    kyy = gaussian_kernel(y, y, sigma)
    kxy = gaussian_kernel(x, y, sigma)
    n = x.shape[0]
    m = y.shape[0]
    mmd2 = (kxx.sum() - kxx.trace()) / (n * (n - 1)) + \
           (kyy.sum() - kyy.trace()) / (m * (m - 1)) - \
           2 * kxy.sum() / (n * m)
    return mmd2


sim_t = torch.from_numpy(sim_features)
real_t = torch.from_numpy(real_features)
adapted_t = torch.from_numpy(adapted_features)

mmd_before = compute_mmd(sim_t, real_t, sigma=2.0).item()
mmd_after = compute_mmd(adapted_t, real_t, sigma=2.0).item()

print(f"MMD\u00b2 (Sim vs Real, before adaptation): {mmd_before:.4f}")
print(f"MMD\u00b2 (Adapted vs Real, after adaptation): {mmd_after:.4f}")
print(f"Reduction: {(1 - mmd_after / mmd_before) * 100:.1f}%")

---

## 3. Domain Randomization for Vision

Domain randomization floods the training pipeline with visually diverse synthetic images by sampling rendering parameters $\xi \sim \Xi$:

$$x_{\text{rand}} = \mathcal{T}(x_{\text{base}};\; \xi), \quad \xi = (\xi_{\text{light}},\; \xi_{\text{noise}},\; \xi_{\text{blur}},\; \xi_{\text{color}},\; \xi_{\text{texture}})$$

**Randomization axes:**
- **Lighting**: direction, intensity, color temperature
- **Noise**: Gaussian, salt-and-pepper, speckle
- **Blur**: motion, defocus, Gaussian
- **Color/contrast**: gamma, channel shifts, saturation
- **Texture**: Perlin noise overlay, frequency modulation

If the randomization distribution $\Xi$ is rich enough:

$$P_{\text{real}}(x) \subset \text{support}\left(\mathbb{E}_{\xi \sim \Xi}[P_{\text{sim}}(x | \xi)]\right)$$

then the real domain becomes a special case the policy has already encountered.

In [ ]:
# --- Domain Randomization: create diverse training images ---

def apply_domain_randomization(img, rng=None):
    """Apply random visual perturbations to a clean sim image."""
    if rng is None:
        rng = np.random.default_rng()
    h, w = img.shape
    out = img.copy()

    # Random brightness & contrast
    alpha = rng.uniform(0.6, 1.4)  # contrast
    beta = rng.uniform(-0.2, 0.2)  # brightness
    out = alpha * out + beta

    # Random lighting gradient
    yy, xx = np.mgrid[:h, :w]
    angle = rng.uniform(0, 2 * np.pi)
    gradient = (np.cos(angle) * xx / w + np.sin(angle) * yy / h)
    grad_strength = rng.uniform(0, 0.25)
    out = out + gradient * grad_strength

    # Random noise (Gaussian or salt-and-pepper)
    noise_type = rng.choice(['gaussian', 'sp', 'speckle'])
    if noise_type == 'gaussian':
        std = rng.uniform(0.02, 0.18)
        out = out + rng.standard_normal((h, w)).astype(np.float32) * std
    elif noise_type == 'sp':
        prob = rng.uniform(0.01, 0.08)
        salt = rng.random((h, w)) < prob / 2
        pepper = rng.random((h, w)) < prob / 2
        out[salt] = 1.0
        out[pepper] = 0.0
    else:  # speckle
        std = rng.uniform(0.05, 0.15)
        out = out + out * rng.standard_normal((h, w)).astype(np.float32) * std

    # Random texture overlay
    if rng.random() > 0.4:
        freq = rng.uniform(3, 15)
        amp = rng.uniform(0.02, 0.1)
        phase = rng.uniform(0, 2 * np.pi)
        tex = amp * np.sin(2 * np.pi * freq * xx / w + phase) * \
              np.cos(2 * np.pi * freq * yy / h + phase)
        out = out + tex

    # Random gamma
    if rng.random() > 0.5:
        gamma = rng.uniform(0.6, 1.6)
        out = np.clip(out, 0, 1)
        out = np.power(out, gamma)

    return np.clip(out, 0, 1).astype(np.float32)


# Show randomization diversity
base_img = make_clean_shape_image(64, 'circle')
fig, axes = plt.subplots(3, 6, figsize=(15, 8))
axes[0, 0].imshow(base_img, cmap='gray', vmin=0, vmax=1)
axes[0, 0].set_title('Original\n(Sim)', fontweight='bold', color='#2196F3')
axes[0, 0].axis('off')

rng = np.random.default_rng(SEED)
for idx in range(1, 18):
    row, col = divmod(idx, 6)
    rand_img = apply_domain_randomization(base_img, rng)
    axes[row, col].imshow(rand_img, cmap='gray', vmin=0, vmax=1)
    axes[row, col].set_title(f'Rand #{idx}', fontsize=9)
    axes[row, col].axis('off')

fig.suptitle('Domain Randomization: One Shape \u2192 Many Visual Variants',
             fontsize=14, fontweight='bold', y=1.0)
plt.tight_layout()
plt.show()

---

## 4. Domain Adaptation Techniques

### 4.1 Adversarial Domain Adaptation

The key idea uses a **Gradient Reversal Layer (GRL)** (Ganin et al., 2016):

During the forward pass the GRL acts as identity; during the backward pass it **negates** the gradient:

$$\text{GRL}(x) = x \quad (\text{forward})$$
$$\frac{\partial \text{GRL}}{\partial x} = -\lambda \, I \quad (\text{backward})$$

This forces the feature extractor $\phi$ to produce **domain-invariant** features: the domain classifier $D$ tries to distinguish sim from real, while $\phi$ tries to make them indistinguishable.

### 4.2 Feature Alignment Objective

The domain discriminator loss (binary cross-entropy):

$$\mathcal{L}_{\text{domain}} = -\frac{1}{n_s}\sum_{i=1}^{n_s} \log D(\phi(x_i^s)) - \frac{1}{n_t}\sum_{j=1}^{n_t} \log\big(1 - D(\phi(x_j^t))\big)$$

where $s$ = source (sim), $t$ = target (real).

### 4.3 Progressive Domain Transfer

The adaptation strength $\lambda$ follows a schedule:

$$\lambda_p = \frac{2}{1 + \exp(-\gamma \cdot p)} - 1, \qquad p = \frac{\text{epoch}}{\text{total\_epochs}}$$

This ramps from 0 to 1, letting the task loss dominate early training before domain alignment kicks in.

In [ ]:
# --- Visualize the \lambda schedule ---

p = np.linspace(0, 1, 200)
gamma_vals = [2, 5, 10]

fig, ax = plt.subplots(figsize=(8, 4))
for g in gamma_vals:
    lam = 2.0 / (1 + np.exp(-g * p)) - 1
    ax.plot(p, lam, linewidth=2.5, label=f'$\\gamma = {g}$')

ax.set_xlabel('Training progress $p$ (epoch / total_epochs)', fontsize=12)
ax.set_ylabel('Adaptation weight $\\lambda_p$', fontsize=12)
ax.set_title('Progressive Domain Adaptation Schedule', fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

---

## 5. Full Implementation

### 5.1 Dataset Generation

We create three datasets:
1. **Sim** (source): clean geometric shapes (circle=0, square=1, triangle=2)
2. **Sim-Randomized**: same shapes with heavy domain randomization
3. **Real** (target): shapes with realistic noise, texture, and lighting

In [ ]:
# --- Generate datasets ---

IMG_SIZE = 48
N_PER_CLASS = 300
SHAPES = ['circle', 'square', 'triangle']
N_CLASSES = len(SHAPES)


def generate_dataset(n_per_class, img_size, domain='sim', seed=0):
    """Generate a dataset of shape images for a given domain."""
    rng = np.random.default_rng(seed)
    images, labels = [], []
    for cls_idx, shape in enumerate(SHAPES):
        for _ in range(n_per_class):
            bg_val = rng.uniform(0.05, 0.25)
            fg_val = rng.uniform(0.7, 0.95)
            img = make_clean_shape_image(img_size, shape, bg_val, fg_val)
            if domain == 'sim_rand':
                img = apply_domain_randomization(img, rng)
            elif domain == 'real':
                img = add_real_effects(
                    img,
                    noise_std=rng.uniform(0.06, 0.14),
                    blur_k=rng.choice([1, 3]),
                    brightness_shift=rng.uniform(-0.1, 0.1),
                    texture_freq=rng.uniform(4, 12),
                    texture_amp=rng.uniform(0.03, 0.1),
                )
            images.append(img)
            labels.append(cls_idx)
    images = np.array(images, dtype=np.float32)
    labels = np.array(labels, dtype=np.int64)
    perm = rng.permutation(len(labels))
    return images[perm], labels[perm]


X_sim, y_sim = generate_dataset(N_PER_CLASS, IMG_SIZE, domain='sim', seed=10)
X_rand, y_rand = generate_dataset(N_PER_CLASS, IMG_SIZE, domain='sim_rand', seed=20)
X_real, y_real = generate_dataset(N_PER_CLASS, IMG_SIZE, domain='real', seed=30)

print(f"Sim dataset:  {X_sim.shape}, labels: {np.bincount(y_sim)}")
print(f"Rand dataset: {X_rand.shape}, labels: {np.bincount(y_rand)}")
print(f"Real dataset: {X_real.shape}, labels: {np.bincount(y_real)}")

In [ ]:
# --- Show sample images from each domain ---

fig, axes = plt.subplots(3, 6, figsize=(14, 7))
domain_data = [
    ('Sim (Clean)', X_sim, y_sim, '#2196F3'),
    ('Sim-Randomized', X_rand, y_rand, '#FF9800'),
    ('Real (Target)', X_real, y_real, '#F44336'),
]

for row, (name, X, y, color) in enumerate(domain_data):
    for col in range(6):
        idx = col * (len(y) // 6)
        axes[row, col].imshow(X[idx], cmap='gray', vmin=0, vmax=1)
        axes[row, col].set_title(f'{SHAPES[y[idx]]}', fontsize=9)
        axes[row, col].axis('off')
    axes[row, 0].set_ylabel(name, fontsize=12, fontweight='bold',
                            color=color, rotation=0, labelpad=80, va='center')

fig.suptitle('Three Domains: Sim \u2192 Randomized \u2192 Real',
             fontsize=14, fontweight='bold', y=1.0)
plt.tight_layout()
plt.show()

### 5.2 Neural Network Architecture

We build a **Domain-Adversarial Neural Network (DANN)** with three components:

1. **Feature Extractor** $G_f$: shared CNN backbone
2. **Shape Classifier** $G_y$: predicts circle / square / triangle
3. **Domain Discriminator** $G_d$: predicts sim vs. real (with gradient reversal)

In [ ]:
# --- Gradient Reversal Layer ---

class GradientReversalFn(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.alpha * grad_output, None


class GradientReversal(nn.Module):
    def __init__(self, alpha=1.0):
        super().__init__()
        self.alpha = alpha

    def forward(self, x):
        return GradientReversalFn.apply(x, self.alpha)


# --- Feature Extractor ---

class FeatureExtractor(nn.Module):
    def __init__(self, feature_dim=64):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.AdaptiveAvgPool2d(4),
        )
        self.fc = nn.Sequential(
            nn.Linear(128 * 4 * 4, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, feature_dim),
        )

    def forward(self, x):
        x = self.conv(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)


# --- Shape Classifier ---

class ShapeClassifier(nn.Module):
    def __init__(self, feature_dim=64, n_classes=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(feature_dim, 32), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(32, n_classes),
        )

    def forward(self, x):
        return self.net(x)


# --- Domain Discriminator ---

class DomainDiscriminator(nn.Module):
    def __init__(self, feature_dim=64):
        super().__init__()
        self.grl = GradientReversal()
        self.net = nn.Sequential(
            nn.Linear(feature_dim, 32), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(32, 1),
        )

    def forward(self, x, alpha=1.0):
        self.grl.alpha = alpha
        x = self.grl(x)
        return self.net(x)


FEAT_DIM = 64
print("Architecture defined: FeatureExtractor + ShapeClassifier + DomainDiscriminator")
print(f"Feature dimension: {FEAT_DIM}")

fe_test = FeatureExtractor(FEAT_DIM).to(device)
total_params = sum(p.numel() for p in fe_test.parameters())
print(f"Feature extractor parameters: {total_params:,}")

### 5.3 Training Utilities and RL Environment

In [ ]:
# --- Prepare torch datasets ---

def to_tensor_dataset(X, y):
    X_t = torch.from_numpy(X).unsqueeze(1).to(device)   # (N, 1, H, W)
    y_t = torch.from_numpy(y).to(device)
    return TensorDataset(X_t, y_t)


# Train/test splits
split = int(0.8 * len(y_sim))
ds_sim_train = to_tensor_dataset(X_sim[:split], y_sim[:split])
ds_sim_test = to_tensor_dataset(X_sim[split:], y_sim[split:])
ds_rand_train = to_tensor_dataset(X_rand[:split], y_rand[:split])
ds_rand_test = to_tensor_dataset(X_rand[split:], y_rand[split:])
ds_real_train = to_tensor_dataset(X_real[:split], y_real[:split])
ds_real_test = to_tensor_dataset(X_real[split:], y_real[split:])

BATCH = 64
loader_sim = DataLoader(ds_sim_train, batch_size=BATCH, shuffle=True)
loader_rand = DataLoader(ds_rand_train, batch_size=BATCH, shuffle=True)
loader_real = DataLoader(ds_real_train, batch_size=BATCH, shuffle=True)

print(f"Train samples per domain: {split}")
print(f"Test  samples per domain: {len(y_sim) - split}")

In [ ]:
# --- Simple RL environment: classify shape and get reward ---

class ShapeClassificationEnv:
    """Image classification as an RL problem.
    State: image, Action: class prediction, Reward: +1 correct, -0.5 wrong.
    """

    def __init__(self, images, labels, feature_extractor):
        self.images = images
        self.labels = labels
        self.fe = feature_extractor
        self.idx = 0
        self.order = np.arange(len(labels))

    def reset(self):
        np.random.shuffle(self.order)
        self.idx = 0
        return self._get_state()

    def _get_state(self):
        img = self.images[self.order[self.idx]]
        with torch.no_grad():
            img_t = torch.from_numpy(img).unsqueeze(0).unsqueeze(0).to(
                next(self.fe.parameters()).device)
            feat = self.fe(img_t)
        return feat.squeeze(0)

    def step(self, action):
        label = self.labels[self.order[self.idx]]
        reward = 1.0 if action == label else -0.5
        self.idx += 1
        done = self.idx >= len(self.labels)
        next_state = None if done else self._get_state()
        return next_state, reward, done, {'label': label}


# --- Simple policy network (RL agent) ---

class PolicyNet(nn.Module):
    def __init__(self, feat_dim=64, n_actions=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(feat_dim, 32), nn.ReLU(),
            nn.Linear(32, n_actions),
        )

    def forward(self, x):
        return F.softmax(self.net(x), dim=-1)

    def act(self, state, greedy=False):
        probs = self.forward(state)
        if greedy:
            return probs.argmax(dim=-1).item()
        return torch.multinomial(probs, 1).item()


print("Environment and PolicyNet defined.")

### 5.4 Training the Three Approaches

We train and compare:

| Approach | Training Data | Domain Adaptation |
|----------|--------------|-------------------|
| **Baseline** | Sim only | None |
| **Domain Randomization** | Sim-Randomized | None |
| **DANN** | Sim + Real (unlabeled) | Adversarial |

In [ ]:
# --- Training functions ---

def train_classifier(feat_ext, classifier, train_loader, n_epochs=25, lr=1e-3):
    """Train a feature extractor + classifier on labelled source data."""
    feat_ext.train()
    classifier.train()
    optimizer = optim.Adam(
        list(feat_ext.parameters()) + list(classifier.parameters()), lr=lr)
    criterion = nn.CrossEntropyLoss()
    history = {'loss': [], 'acc': []}

    for epoch in range(n_epochs):
        total_loss, correct, total = 0, 0, 0
        for xb, yb in train_loader:
            feats = feat_ext(xb)
            logits = classifier(feats)
            loss = criterion(logits, yb)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * xb.size(0)
            correct += (logits.argmax(1) == yb).sum().item()
            total += xb.size(0)
        history['loss'].append(total_loss / total)
        history['acc'].append(correct / total)
    return history


def train_dann(feat_ext, classifier, discriminator,
               source_loader, target_loader, n_epochs=30, lr=1e-3, gamma=10):
    """Domain-adversarial training with GRL."""
    feat_ext.train()
    classifier.train()
    discriminator.train()
    all_params = list(feat_ext.parameters()) + list(classifier.parameters()) + \
                 list(discriminator.parameters())
    optimizer = optim.Adam(all_params, lr=lr)
    cls_criterion = nn.CrossEntropyLoss()
    dom_criterion = nn.BCEWithLogitsLoss()
    history = {'task_loss': [], 'domain_loss': [], 'src_acc': [], 'alpha': []}

    for epoch in range(n_epochs):
        p = epoch / n_epochs
        alpha = float(2.0 / (1 + np.exp(-gamma * p)) - 1)
        total_task, total_dom, correct, total = 0, 0, 0, 0

        target_iter = iter(target_loader)
        for xb_s, yb_s in source_loader:
            try:
                xb_t, _ = next(target_iter)
            except StopIteration:
                target_iter = iter(target_loader)
                xb_t, _ = next(target_iter)

            bs_s, bs_t = xb_s.size(0), xb_t.size(0)

            feats_s = feat_ext(xb_s)
            feats_t = feat_ext(xb_t)

            # Task loss (source only)
            logits_s = classifier(feats_s)
            task_loss = cls_criterion(logits_s, yb_s)

            # Domain loss (both)
            dom_s = discriminator(feats_s, alpha)
            dom_t = discriminator(feats_t, alpha)
            dom_labels = torch.cat([
                torch.zeros(bs_s, 1, device=device),
                torch.ones(bs_t, 1, device=device)
            ])
            dom_preds = torch.cat([dom_s, dom_t])
            domain_loss = dom_criterion(dom_preds, dom_labels)

            loss = task_loss + domain_loss
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_task += task_loss.item() * bs_s
            total_dom += domain_loss.item() * (bs_s + bs_t)
            correct += (logits_s.argmax(1) == yb_s).sum().item()
            total += bs_s

        history['task_loss'].append(total_task / total)
        history['domain_loss'].append(total_dom / (total * 2))
        history['src_acc'].append(correct / total)
        history['alpha'].append(alpha)

    return history


@torch.no_grad()
def evaluate(feat_ext, classifier, dataset):
    feat_ext.eval()
    classifier.eval()
    loader = DataLoader(dataset, batch_size=256)
    correct, total = 0, 0
    for xb, yb in loader:
        logits = classifier(feat_ext(xb))
        correct += (logits.argmax(1) == yb).sum().item()
        total += xb.size(0)
    return correct / total


print("Training utilities ready.")

In [ ]:
# === Approach 1: Baseline (train on clean sim only) ===
torch.manual_seed(SEED)
fe_baseline = FeatureExtractor(FEAT_DIM).to(device)
cls_baseline = ShapeClassifier(FEAT_DIM, N_CLASSES).to(device)
hist_baseline = train_classifier(fe_baseline, cls_baseline, loader_sim, n_epochs=30)

acc_bl_sim = evaluate(fe_baseline, cls_baseline, ds_sim_test)
acc_bl_real = evaluate(fe_baseline, cls_baseline, ds_real_test)
print(f"[Baseline]  Sim acc: {acc_bl_sim:.3f}  |  Real acc: {acc_bl_real:.3f}  |  "
      f"Transfer gap: {acc_bl_sim - acc_bl_real:.3f}")

In [ ]:
# === Approach 2: Domain Randomization ===
torch.manual_seed(SEED)
fe_rand = FeatureExtractor(FEAT_DIM).to(device)
cls_rand = ShapeClassifier(FEAT_DIM, N_CLASSES).to(device)
hist_rand = train_classifier(fe_rand, cls_rand, loader_rand, n_epochs=30)

acc_dr_sim = evaluate(fe_rand, cls_rand, ds_sim_test)
acc_dr_real = evaluate(fe_rand, cls_rand, ds_real_test)
print(f"[DomRand]   Sim acc: {acc_dr_sim:.3f}  |  Real acc: {acc_dr_real:.3f}  |  "
      f"Transfer gap: {acc_dr_sim - acc_dr_real:.3f}")

In [ ]:
# === Approach 3: DANN (adversarial domain adaptation) ===
torch.manual_seed(SEED)
fe_dann = FeatureExtractor(FEAT_DIM).to(device)
cls_dann = ShapeClassifier(FEAT_DIM, N_CLASSES).to(device)
disc_dann = DomainDiscriminator(FEAT_DIM).to(device)

hist_dann = train_dann(
    fe_dann, cls_dann, disc_dann,
    source_loader=loader_sim,
    target_loader=loader_real,
    n_epochs=35, lr=1e-3, gamma=10,
)

acc_da_sim = evaluate(fe_dann, cls_dann, ds_sim_test)
acc_da_real = evaluate(fe_dann, cls_dann, ds_real_test)
print(f"[DANN]      Sim acc: {acc_da_sim:.3f}  |  Real acc: {acc_da_real:.3f}  |  "
      f"Transfer gap: {acc_da_sim - acc_da_real:.3f}")

In [ ]:
# --- Training curves comparison ---

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# Loss curves
axes[0].plot(hist_baseline['loss'], label='Baseline (Sim)', linewidth=2, color='#9E9E9E')
axes[0].plot(hist_rand['loss'], label='Dom. Rand.', linewidth=2, color='#FF9800')
axes[0].plot(hist_dann['task_loss'], label='DANN (task)', linewidth=2, color='#4CAF50')
axes[0].plot(hist_dann['domain_loss'], label='DANN (domain)', linewidth=2,
             linestyle='--', color='#4CAF50', alpha=0.6)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss', fontweight='bold')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# Source accuracy
axes[1].plot(hist_baseline['acc'], label='Baseline', linewidth=2, color='#9E9E9E')
axes[1].plot(hist_rand['acc'], label='Dom. Rand.', linewidth=2, color='#FF9800')
axes[1].plot(hist_dann['src_acc'], label='DANN', linewidth=2, color='#4CAF50')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Source Accuracy')
axes[1].set_title('Training Accuracy (Source)', fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(0, 1.05)

# Adaptation schedule
axes[2].plot(hist_dann['alpha'], linewidth=2.5, color='#E91E63')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('$\\alpha$ (GRL strength)')
axes[2].set_title('DANN Adaptation Schedule', fontweight='bold')
axes[2].grid(True, alpha=0.3)
axes[2].set_ylim(0, 1.05)

fig.suptitle('Training Dynamics Across Approaches', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 5.5 RL Agent: Train in Sim, Test in Real

In [ ]:
# --- Train RL policy on top of each feature extractor ---

def train_rl_policy(feat_ext, train_images, train_labels, n_episodes=60,
                    lr=3e-3, gamma_rl=0.99):
    """REINFORCE policy gradient on the shape classification task."""
    feat_ext.eval()
    policy = PolicyNet(FEAT_DIM, N_CLASSES).to(device)
    optimizer = optim.Adam(policy.parameters(), lr=lr)
    env = ShapeClassificationEnv(train_images, train_labels, feat_ext)
    reward_history = []

    for ep in range(n_episodes):
        state = env.reset()
        log_probs, rewards = [], []
        done = False
        steps = 0
        max_steps = min(200, len(train_labels))

        while not done and steps < max_steps:
            probs = policy(state)
            dist = torch.distributions.Categorical(probs)
            action = dist.sample()
            log_probs.append(dist.log_prob(action))
            state, reward, done, _ = env.step(action.item())
            rewards.append(reward)
            steps += 1

        # Compute discounted returns
        returns = []
        G = 0
        for r in reversed(rewards):
            G = r + gamma_rl * G
            returns.insert(0, G)
        returns = torch.tensor(returns, device=device)
        if returns.std() > 1e-6:
            returns = (returns - returns.mean()) / (returns.std() + 1e-8)

        loss = -sum(lp * G for lp, G in zip(log_probs, returns))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        reward_history.append(np.mean(rewards))

    return policy, reward_history


def eval_rl_policy(feat_ext, policy, images, labels):
    """Evaluate RL policy accuracy on a dataset."""
    feat_ext.eval()
    policy.eval()
    env = ShapeClassificationEnv(images, labels, feat_ext)
    state = env.reset()
    correct, total = 0, 0
    done = False
    while not done and total < len(labels):
        action = policy.act(state, greedy=True)
        state, reward, done, info = env.step(action)
        correct += int(action == info['label'])
        total += 1
    return correct / total


# Train RL policies
print("Training RL agents...")
torch.manual_seed(SEED)
pol_baseline, rh_baseline = train_rl_policy(fe_baseline, X_sim[:split], y_sim[:split])
print("  Baseline RL trained.")

torch.manual_seed(SEED)
pol_rand, rh_rand = train_rl_policy(fe_rand, X_rand[:split], y_rand[:split])
print("  Domain Randomization RL trained.")

torch.manual_seed(SEED)
pol_dann, rh_dann = train_rl_policy(fe_dann, X_sim[:split], y_sim[:split])
print("  DANN RL trained.")

In [ ]:
# --- RL training reward curves ---

fig, ax = plt.subplots(figsize=(10, 5))
window = 5

for rh, label, color in [
    (rh_baseline, 'Baseline', '#9E9E9E'),
    (rh_rand, 'Domain Rand.', '#FF9800'),
    (rh_dann, 'DANN', '#4CAF50'),
]:
    smoothed = np.convolve(rh, np.ones(window)/window, mode='valid')
    ax.plot(smoothed, label=label, linewidth=2.5, color=color)
    ax.fill_between(range(len(smoothed)), smoothed - 0.05, smoothed + 0.05,
                    alpha=0.1, color=color)

ax.set_xlabel('Episode', fontsize=12)
ax.set_ylabel('Mean Episode Reward', fontsize=12)
ax.set_title('RL Policy Training Curves (Sim Domain)', fontweight='bold', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# --- Evaluate RL policies on both domains ---

results = {}
for name, fe, pol in [
    ('Baseline', fe_baseline, pol_baseline),
    ('DomRand', fe_rand, pol_rand),
    ('DANN', fe_dann, pol_dann),
]:
    sim_acc = eval_rl_policy(fe, pol, X_sim[split:], y_sim[split:])
    real_acc = eval_rl_policy(fe, pol, X_real[split:], y_real[split:])
    results[name] = {'sim': sim_acc, 'real': real_acc, 'gap': sim_acc - real_acc}
    print(f"[{name:10s}]  Sim: {sim_acc:.3f}  Real: {real_acc:.3f}  Gap: {sim_acc - real_acc:+.3f}")

# Bar chart comparison
fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(3)
width = 0.3
names = list(results.keys())
sim_accs = [results[n]['sim'] for n in names]
real_accs = [results[n]['real'] for n in names]
gaps = [results[n]['gap'] for n in names]

bars1 = ax.bar(x - width/2, sim_accs, width, label='Sim Test', color='#2196F3', alpha=0.85)
bars2 = ax.bar(x + width/2, real_accs, width, label='Real Test', color='#F44336', alpha=0.85)

for i, (b1, b2) in enumerate(zip(bars1, bars2)):
    ax.annotate(f'{sim_accs[i]:.2f}', xy=(b1.get_x() + b1.get_width()/2, b1.get_height()),
               ha='center', va='bottom', fontsize=11, fontweight='bold')
    ax.annotate(f'{real_accs[i]:.2f}', xy=(b2.get_x() + b2.get_width()/2, b2.get_height()),
               ha='center', va='bottom', fontsize=11, fontweight='bold')
    mid_x = (b1.get_x() + b2.get_x() + b2.get_width()) / 2
    ax.annotate(f'Gap: {gaps[i]:+.2f}', xy=(mid_x, 0.02),
               ha='center', fontsize=10, color='gray', style='italic')

ax.set_xlabel('Approach', fontsize=12)
ax.set_ylabel('RL Policy Accuracy', fontsize=12)
ax.set_title('Sim-to-Real Transfer: RL Policy Performance', fontweight='bold', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(names, fontsize=12)
ax.legend(fontsize=11)
ax.set_ylim(0, 1.15)
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

---

## 6. Analysis

### 6.1 Feature Space Visualization (t-SNE)

In [ ]:
# --- Extract features for t-SNE ---

@torch.no_grad()
def extract_features(feat_ext, X):
    feat_ext.eval()
    X_t = torch.from_numpy(X).unsqueeze(1).to(device)
    feats = []
    for i in range(0, len(X_t), 256):
        feats.append(feat_ext(X_t[i:i+256]).cpu().numpy())
    return np.vstack(feats)


n_vis = 200
feats_bl_sim = extract_features(fe_baseline, X_sim[:n_vis])
feats_bl_real = extract_features(fe_baseline, X_real[:n_vis])
feats_dr_sim = extract_features(fe_rand, X_sim[:n_vis])
feats_dr_real = extract_features(fe_rand, X_real[:n_vis])
feats_da_sim = extract_features(fe_dann, X_sim[:n_vis])
feats_da_real = extract_features(fe_dann, X_real[:n_vis])

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

for ax, title, f_sim, f_real, labs_sim, labs_real in [
    (axes[0], 'Baseline', feats_bl_sim, feats_bl_real, y_sim[:n_vis], y_real[:n_vis]),
    (axes[1], 'Domain Randomization', feats_dr_sim, feats_dr_real, y_sim[:n_vis], y_real[:n_vis]),
    (axes[2], 'DANN (Adapted)', feats_da_sim, feats_da_real, y_sim[:n_vis], y_real[:n_vis]),
]:
    combined = np.vstack([f_sim, f_real])
    domains = np.array([0]*n_vis + [1]*n_vis)
    labels = np.concatenate([labs_sim, labs_real])

    tsne = TSNE(n_components=2, perplexity=30, random_state=SEED, n_iter=800)
    emb = tsne.fit_transform(combined)

    colors_shape = ['#2196F3', '#FF9800', '#4CAF50']
    markers = ['o', 'x']
    for d, marker, dom_name in [(0, 'o', 'Sim'), (1, 'x', 'Real')]:
        mask_d = domains == d
        for c in range(N_CLASSES):
            mask = mask_d & (labels == c)
            ax.scatter(emb[mask, 0], emb[mask, 1], c=colors_shape[c],
                       marker=marker, s=20, alpha=0.6,
                       label=f'{dom_name}-{SHAPES[c]}' if d == 0 or c == 0 else None)
    ax.set_title(title, fontweight='bold', fontsize=13)
    ax.set_xlabel('t-SNE 1')
    ax.set_ylabel('t-SNE 2')
    ax.grid(True, alpha=0.2)

# Legend
handles = [
    plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='gray',
               markersize=8, label='Sim domain'),
    plt.Line2D([0], [0], marker='x', color='gray', markerfacecolor='gray',
               markersize=8, label='Real domain', linestyle='None'),
]
for c in range(N_CLASSES):
    handles.append(mpatches.Patch(color=colors_shape[c], label=SHAPES[c].capitalize()))
fig.legend(handles=handles, loc='lower center', ncol=5, fontsize=10,
           bbox_to_anchor=(0.5, -0.05))

fig.suptitle('t-SNE Feature Visualization: Domain Alignment',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 6.2 MMD Analysis Across Approaches

In [ ]:
# --- Compute MMD for each approach ---

mmds = {}
for name, f_sim, f_real in [
    ('Baseline', feats_bl_sim, feats_bl_real),
    ('DomRand', feats_dr_sim, feats_dr_real),
    ('DANN', feats_da_sim, feats_da_real),
]:
    s_t = torch.from_numpy(f_sim[:150])
    r_t = torch.from_numpy(f_real[:150])
    mmd_val = compute_mmd(s_t, r_t, sigma=5.0).item()
    mmds[name] = max(mmd_val, 0)
    print(f"{name:12s} MMD\u00b2: {mmd_val:.5f}")

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#9E9E9E', '#FF9800', '#4CAF50']
bars = ax.bar(mmds.keys(), mmds.values(), color=colors, edgecolor='black', linewidth=0.8)
for bar, val in zip(bars, mmds.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
            f'{val:.4f}', ha='center', va='bottom', fontweight='bold', fontsize=12)
ax.set_ylabel('MMD\u00b2 (Sim vs Real features)', fontsize=12)
ax.set_title('Domain Gap Measured by MMD\u00b2', fontweight='bold', fontsize=14)
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

### 6.3 Randomization Parameter Sensitivity

In [ ]:
# --- Sensitivity: how does randomization breadth affect transfer? ---

def generate_scaled_rand(n_per_class, img_size, scale=1.0, seed=0):
    """Generate domain-randomized data with controllable randomization intensity."""
    rng = np.random.default_rng(seed)
    images, labels = [], []
    for cls_idx, shape in enumerate(SHAPES):
        for _ in range(n_per_class):
            bg_val = rng.uniform(0.05, 0.25)
            fg_val = rng.uniform(0.7, 0.95)
            img = make_clean_shape_image(img_size, shape, bg_val, fg_val)
            h, w = img.shape
            alpha = 1 + (rng.uniform(-0.4, 0.4)) * scale
            beta = rng.uniform(-0.2, 0.2) * scale
            img = alpha * img + beta
            noise_std = rng.uniform(0.02, 0.18) * scale
            img = img + rng.standard_normal((h, w)).astype(np.float32) * noise_std
            images.append(np.clip(img, 0, 1).astype(np.float32))
            labels.append(cls_idx)
    perm = rng.permutation(len(labels))
    return np.array(images)[perm], np.array(labels)[perm]


scales = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0, 1.5, 2.0]
real_accs_sweep = []
sim_accs_sweep = []

for sc in scales:
    X_sc, y_sc = generate_scaled_rand(N_PER_CLASS, IMG_SIZE, scale=sc, seed=100)
    ds_sc = to_tensor_dataset(X_sc[:split], y_sc[:split])
    loader_sc = DataLoader(ds_sc, batch_size=BATCH, shuffle=True)

    torch.manual_seed(SEED + int(sc * 100))
    fe_sc = FeatureExtractor(FEAT_DIM).to(device)
    cls_sc = ShapeClassifier(FEAT_DIM, N_CLASSES).to(device)
    train_classifier(fe_sc, cls_sc, loader_sc, n_epochs=20)

    sim_accs_sweep.append(evaluate(fe_sc, cls_sc, ds_sim_test))
    real_accs_sweep.append(evaluate(fe_sc, cls_sc, ds_real_test))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(scales, sim_accs_sweep, 'o-', linewidth=2.5, markersize=8,
        color='#2196F3', label='Sim Test Acc')
ax.plot(scales, real_accs_sweep, 's-', linewidth=2.5, markersize=8,
        color='#F44336', label='Real Test Acc')
ax.fill_between(scales, sim_accs_sweep, real_accs_sweep, alpha=0.15, color='gray',
                label='Transfer Gap')

ax.set_xlabel('Randomization Scale', fontsize=12)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Randomization Intensity vs. Transfer Performance',
             fontweight='bold', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

print("\nKey insight: moderate randomization (0.6\u20131.0) often yields the best transfer.")
print("Too little \u2192 sim-only overfitting; too much \u2192 training signal is washed out.")

### 6.4 Before / After on Real-Domain Images

In [ ]:
# --- Before/after: show predictions on real images ---

@torch.no_grad()
def predict_batch(feat_ext, classifier, images):
    feat_ext.eval()
    classifier.eval()
    X_t = torch.from_numpy(images).unsqueeze(1).to(device)
    feats = feat_ext(X_t)
    logits = classifier(feats)
    probs = F.softmax(logits, dim=1)
    preds = logits.argmax(1).cpu().numpy()
    confs = probs.max(1).values.cpu().numpy()
    return preds, confs


n_show = 8
show_idx = np.random.choice(len(y_real) - split, n_show, replace=False)
show_images = X_real[split:][show_idx]
show_labels = y_real[split:][show_idx]

preds_bl, confs_bl = predict_batch(fe_baseline, cls_baseline, show_images)
preds_da, confs_da = predict_batch(fe_dann, cls_dann, show_images)

fig, axes = plt.subplots(3, n_show, figsize=(16, 7))
for col in range(n_show):
    axes[0, col].imshow(show_images[col], cmap='gray', vmin=0, vmax=1)
    axes[0, col].set_title(f'GT: {SHAPES[show_labels[col]]}', fontsize=9, fontweight='bold')
    axes[0, col].axis('off')

    is_correct_bl = preds_bl[col] == show_labels[col]
    axes[1, col].imshow(show_images[col], cmap='gray', vmin=0, vmax=1)
    axes[1, col].set_title(
        f'{SHAPES[preds_bl[col]]} ({confs_bl[col]:.0%})',
        fontsize=9, color='green' if is_correct_bl else 'red', fontweight='bold')
    axes[1, col].axis('off')

    is_correct_da = preds_da[col] == show_labels[col]
    axes[2, col].imshow(show_images[col], cmap='gray', vmin=0, vmax=1)
    axes[2, col].set_title(
        f'{SHAPES[preds_da[col]]} ({confs_da[col]:.0%})',
        fontsize=9, color='green' if is_correct_da else 'red', fontweight='bold')
    axes[2, col].axis('off')

axes[0, 0].set_ylabel('Real Image', fontsize=11, fontweight='bold', rotation=0,
                      labelpad=70, va='center')
axes[1, 0].set_ylabel('Baseline\n(Before)', fontsize=11, fontweight='bold',
                      color='#9E9E9E', rotation=0, labelpad=70, va='center')
axes[2, 0].set_ylabel('DANN\n(After)', fontsize=11, fontweight='bold',
                      color='#4CAF50', rotation=0, labelpad=70, va='center')

fig.suptitle('Before vs. After Domain Adaptation on Real Images',
             fontsize=14, fontweight='bold', y=1.0)
plt.tight_layout()
plt.show()

### 6.5 Transfer Gap Analysis

In [ ]:
# --- Comprehensive transfer gap summary ---

fig = plt.figure(figsize=(14, 6))
gs = GridSpec(1, 2, width_ratios=[1.2, 1])

# Panel 1: Classifier + RL accuracy
ax1 = fig.add_subplot(gs[0])
categories = ['Classifier\n(Sim)', 'Classifier\n(Real)', 'RL Policy\n(Sim)', 'RL Policy\n(Real)']
x = np.arange(len(categories))
w = 0.22

bl_vals = [acc_bl_sim, acc_bl_real, results['Baseline']['sim'], results['Baseline']['real']]
dr_vals = [acc_dr_sim, acc_dr_real, results['DomRand']['sim'], results['DomRand']['real']]
da_vals = [acc_da_sim, acc_da_real, results['DANN']['sim'], results['DANN']['real']]

ax1.bar(x - w, bl_vals, w, label='Baseline', color='#9E9E9E', edgecolor='black', linewidth=0.5)
ax1.bar(x, dr_vals, w, label='DomRand', color='#FF9800', edgecolor='black', linewidth=0.5)
ax1.bar(x + w, da_vals, w, label='DANN', color='#4CAF50', edgecolor='black', linewidth=0.5)

ax1.set_xticks(x)
ax1.set_xticklabels(categories, fontsize=10)
ax1.set_ylabel('Accuracy', fontsize=12)
ax1.set_title('Complete Transfer Performance', fontweight='bold', fontsize=13)
ax1.legend(fontsize=10)
ax1.set_ylim(0, 1.1)
ax1.grid(True, axis='y', alpha=0.3)

# Panel 2: Transfer gap (smaller = better)
ax2 = fig.add_subplot(gs[1])
gap_names = ['Baseline', 'DomRand', 'DANN']
cls_gaps = [acc_bl_sim - acc_bl_real, acc_dr_sim - acc_dr_real, acc_da_sim - acc_da_real]
rl_gaps = [results['Baseline']['gap'], results['DomRand']['gap'], results['DANN']['gap']]

x2 = np.arange(3)
ax2.bar(x2 - 0.15, cls_gaps, 0.3, label='Classifier Gap', color='#E91E63', alpha=0.8)
ax2.bar(x2 + 0.15, rl_gaps, 0.3, label='RL Policy Gap', color='#9C27B0', alpha=0.8)
ax2.set_xticks(x2)
ax2.set_xticklabels(gap_names, fontsize=11)
ax2.set_ylabel('Transfer Gap (lower is better)', fontsize=11)
ax2.set_title('Transfer Gap Comparison', fontweight='bold', fontsize=13)
ax2.legend(fontsize=10)
ax2.grid(True, axis='y', alpha=0.3)
ax2.axhline(0, color='black', linewidth=0.5)

plt.tight_layout()
plt.show()

In [ ]:
# --- PCA feature alignment visualization ---

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, title, f_sim, f_real in [
    (axes[0], 'Baseline', feats_bl_sim, feats_bl_real),
    (axes[1], 'Domain Randomization', feats_dr_sim, feats_dr_real),
    (axes[2], 'DANN', feats_da_sim, feats_da_real),
]:
    combined = np.vstack([f_sim, f_real])
    pca = PCA(n_components=2)
    proj = pca.fit_transform(combined)
    n = len(f_sim)

    ax.scatter(proj[:n, 0], proj[:n, 1], c='#2196F3', s=15, alpha=0.5, label='Sim')
    ax.scatter(proj[n:, 0], proj[n:, 1], c='#F44336', s=15, alpha=0.5, label='Real')

    # Draw covariance ellipses
    for data, color in [(proj[:n], '#2196F3'), (proj[n:], '#F44336')]:
        cov = np.cov(data.T)
        eigvals, eigvecs = np.linalg.eigh(cov)
        angle = np.degrees(np.arctan2(eigvecs[1, 1], eigvecs[0, 1]))
        w, h = 2 * 2 * np.sqrt(eigvals)
        ellipse = mpatches.Ellipse(
            data.mean(0), w, h, angle=angle, fill=False,
            edgecolor=color, linewidth=2, linestyle='--')
        ax.add_patch(ellipse)

    ax.set_title(title, fontweight='bold', fontsize=12)
    ax.legend(fontsize=10)
    ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.0%})')
    ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.0%})')
    ax.grid(True, alpha=0.2)

fig.suptitle('PCA Feature Space with Covariance Ellipses',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---

## 7. Summary

### Key Takeaways

| Concept | Insight |
|---------|--------|
| **Reality Gap** | Policies trained on clean synthetic images fail on real images due to distributional shift $P_{\text{sim}} \neq P_{\text{real}}$ |
| **Domain Randomization** | Broadening simulation diversity via randomized rendering parameters $\xi \sim \Xi$ helps the real domain fall within the training distribution |
| **Domain Adaptation (DANN)** | Adversarial training with a Gradient Reversal Layer forces the feature extractor to produce domain-invariant features |
| **MMD** | Maximum Mean Discrepancy $\|\mu_{\text{sim}} - \mu_{\text{real}}\|_{\mathcal{H}}^2$ quantifies feature distribution gap; lower is better |
| **Progressive Schedule** | Ramping adaptation strength $\lambda_p$ prevents early training instability |
| **Randomization Sweet Spot** | Moderate randomization gives the best sim-to-real transfer; excessive randomization degrades task learning |

### Mathematical Recap

$$\boxed{\pi^* = \arg\max_\pi \; \mathbb{E}_{\xi \sim \Xi}\left[\mathbb{E}_\pi\left[\sum_t \gamma^t r_t \mid \xi\right]\right]}$$

$$\boxed{\mathcal{L}_{\text{adapt}} = \mathcal{L}_{\text{task}} + \lambda \, \mathcal{L}_{\text{domain}}}$$

$$\boxed{\text{MMD}^2 = \|\mu_{\text{sim}} - \mu_{\text{real}}\|_{\mathcal{H}}^2}$$

### Practical Guidelines

1. **Start with domain randomization** — it is simple, effective, and requires no real data
2. **Add domain adaptation** when you have unlabelled real data and need to close the remaining gap
3. **Monitor MMD / t-SNE** to verify that features are aligning
4. **Tune randomization breadth** — too little undercovers the real domain; too much destroys the learning signal
5. **Use progressive scheduling** for stable adversarial training

---

*Module 11.5 — Sim-to-Real Transfer for Vision RL — Complete*